# Track benchmark

Compares how well the sequence-to-function models predict the cardioid tracks on their held-out
test intervals: Pearson r per track over the concatenated test intervals (ATAC, Cut&Tag, RNA) and
gene-level RNA correlations (raw and cell-type-specific, i.e. quantile-normalised), as beeswarms.

**Inputs** (from `config/paths.yaml`), written by the test-evaluation steps of the workflows:

* `model_eval_dir/AlphaGenome_pretrained/<predictions>/track_correlations.csv` (`alphagenome_api/03_test_eval.sh`)
* `model_eval_dir/AlphaGenome_ft/AG_ft_fold1/{1,128}bp_embedding_test_eval_<checkpoint>/track_correlations.csv`
  (`alphagenome_ft/03_test_eval.sh`, `RESOLUTION=1` and `RESOLUTION=128`)
* `model_eval_dir/Borzoi/<predictions>/track_correlations.csv` (`borzoi/03_test_eval.sh`)
* `chrombpnet_model_dir/<cell type>_fold_*/evaluation/chrombpnet_metrics.json` (ChromBPNet peak count
  correlation, averaged over folds)

AlphaGenome (fold 1 test intervals) and Borzoi (`human_rep0` test split) are evaluated on different test
intervals by design; ChromBPNet reports the count correlation in peaks of its own test chromosomes.

In [ ]:
import glob
import json
import os

import pandas as pd
import matplotlib.pyplot as plt

from cardiomics_s2f.config import load_paths
from cardiomics_s2f.eval.track_correlation import plot_correlation_beeswarm

%matplotlib inline

In [ ]:
paths = load_paths()
EVAL_DIR = paths.model_eval_dir


def single_match(pattern):
    """Returns the only file matching a glob pattern (fails if there are none or several)."""
    matches = sorted(glob.glob(str(pattern)))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one file for {pattern}, found {len(matches)}: {matches}")
    return matches[0]


# Test-evaluation results of each model (one run per pattern)
RESULTS = {
    "AG_pre": EVAL_DIR / "AlphaGenome_pretrained" / "*" / "track_correlations.csv",
    "AG_ft_1bp": EVAL_DIR / "AlphaGenome_ft" / "AG_ft_fold1" / "1bp_embedding_test_eval_*" / "track_correlations.csv",
    "AG_ft_128bp": EVAL_DIR / "AlphaGenome_ft" / "AG_ft_fold1" / "128bp_embedding_test_eval_*" / "track_correlations.csv",
    "Borzoi": EVAL_DIR / "Borzoi" / "*" / "track_correlations.csv",
}
CBPN_DIR = paths.chrombpnet_model_dir

In [ ]:
#extract chromBPNet scores
search_pattern = os.path.join(CBPN_DIR, "*_fold_*/evaluation/chrombpnet_metrics.json")
json_files = glob.glob(search_pattern)

data_list = []
for file in json_files:
    parts = os.path.normpath(file).split(os.sep)
    fold_dir = parts[-3]
    if "_fold_" in fold_dir:
        day_celltype, fold_num = fold_dir.rsplit("_fold_", 1)
        try:
            with open(file, 'r') as f:
                metrics = json.load(f)
            pearson_r = metrics["counts_metrics"]["peaks"]["pearsonr"]
            data_list.append({
                "Model": day_celltype,
                "Fold": f"fold_{fold_num}",
                "Pearson R": pearson_r
            })
        except Exception as e:
            print(f"Error reading or parsing {file}: {e}")

cbpn = pd.DataFrame(data_list)

In [ ]:
#load predictions
p = {model: pd.read_csv(single_match(pattern)) for model, pattern in RESULTS.items()}

In [ ]:
atac_labels = []
atac_r = []
for model, res in p.items():
    atac_labels.append(f"{model} ATAC")
    r = res[res.Assay_type == "ATAC"].concat_cor
    atac_r.append(r)

In [ ]:
cbpn = cbpn.groupby("Model", observed=True).agg({"Pearson R": "mean"}).reset_index()
atac_labels.append("chromBPNets")
atac_r.append(cbpn["Pearson R"])

In [ ]:
rna_labels = []
rna_r = []
for model, res in p.items():
    rna_labels.append(f"{model} RNA")
    r = res[res.Assay_type == "RNA"].concat_cor
    rna_r.append(r)

In [ ]:
cnt_labels = []
cnt_r = []
for model, res in p.items():
    r = res[res.Assay_type == "CnT"].concat_cor
    if (len(r) == 0) | (r.isna().all()): continue
    cnt_labels.append(f"{model} CnT")
    cnt_r.append(r)

In [ ]:
gene_labels = []
gene_r = []
for model, res in p.items():
    gene_labels.append(f"{model} Raw Expression")
    gene_r.append(res[res.Assay_type == "RNA"].raw_gene_cor)
for model, res in p.items():
    gene_labels.append(f"{model} Cell Type Specific")
    gene_r.append(res[res.Assay_type == "RNA"].norm_gene_cor)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize = (16, 5), gridspec_kw={'width_ratios': [1, 0.6, 1, 2]})
plot_correlation_beeswarm(values=atac_r, labels=atac_labels, ax = axes[0], title = "scMultiome ATAC")
plot_correlation_beeswarm(values=cnt_r, labels=cnt_labels, ax = axes[1], title = "Cut&TAG")
plot_correlation_beeswarm(values=rna_r, labels=rna_labels, ax = axes[2], title = "scMultiome RNA")
plot_correlation_beeswarm(values=gene_r, labels=gene_labels, ax = axes[3], title = "Gene Expression (Exons counts)")

plt.tight_layout()
plt.show()